# Hardware diagnostics for *Amortized Quantum Policy Evaluation in Reinforcement Learning*

**What this runs.** A four-state reversible chain's symmetrized transition operator is block-encoded (LCU), its
qubitized walk $W$ is built, and the Chebyshev moments $\mu_k^{\pm}=\hat b_\pm^{\top}T_k(S/\lambda_1)\hat b_\pm$
of the paper's two balanced polarization ports are measured on IBM hardware by Hadamard tests of $W^k$,
$k=1,\dots,K$. The paper's own estimator layer (balanced polarization, passive NNLS with the exact
stationary gain pole, kernel-sum comparator) is then run **on the hardware moments**, against an exact
$4\times4$ reference.

**Why this strengthens the paper.** The paper's first stated limitation is that all data are classically
simulated under an i.i.d. Gaussian per-moment noise proxy, with no hardware. This experiment replaces that
proxy with measured device noise for the estimator layer and produces two pre-specified datasets:

- **Q1 — per-moment bias and variance vs circuit volume**: the measured decay of $\mu_k$ with transpiled
  two-qubit gate count (the moment-bias-vs-circuit-volume profile), and how far real noise departs from the
  Gaussian proxy (bias vs shot noise).
- **Q2 — estimator robustness on real moments**: value-curve reconstruction from hardware moments through
  the passive pipeline vs the raw kernel sum, raw and under a transparent one-parameter depolarizing
  self-calibration; plus an exact-moment control arm that isolates the truncation floor.

**Honest scope (state this wherever results are used).** $N=4$ is classically trivial; nothing here is an
advantage or speedup demonstration, and the deep-horizon conditioning regime ($K_r$ up to $4{,}528$) remains
simulation-only. This is a hardware validation of the *estimator layer* and of the noise model — the
diagnostics posture, exactly as the paper's outlook describes.

**Costs before you run.** The whole experiment is $2\times K$ Hadamard-test circuits ($20$ at the default
$K=10$) at $8{,}192$ shots, submitted as **one batched job** — one queue wait, roughly **1–2 minutes of the
Open Plan's 10 free QPU minutes per 28 days**. The run cell checks your recent usage first and refuses to
start if the window is nearly spent. Validate the whole flow with `RUN_ON_HARDWARE = False` before spending any of it.

**Hardware record so far (run 1, `ibm_fez`, 2026-07-22).** The deep instance measured the decoherence
wall, not spectroscopy: only $k=1$ survived (36% signal at 123 routed gates; retention $0.9917$/2q gate,
port-consistent to $3\times10^{-5}$), everything deeper decohered to a small negative offset, and the wall
sits at $\approx480$ routed two-qubit gates. Shot variance matched the model exactly
($0.01104$ vs $\sqrt{1/8192}=0.01105$). That run's curve numbers are **barred** as accuracy claims; its
wall dataset and variance validation go in the paper. The default is now the **lite** instance
($\approx6$ two-qubit gates per walk step, 3 qubits), which at the fez-measured noise level keeps **all
ten moment orders** above the $5\sigma$ gate and reconstructs the curve to $\approx1\%$ in simulation —
spectroscopy is expected to work on the same device.

**Run 2 (lite, `ibm_fez`, 2026-07-22): genuine spectroscopy.** All ten Chebyshev orders on both balanced
ports cleared the sign-consistent $5\sigma$ gate at 7–97 routed two-qubit gates (sign-perfect on the
oscillating port). Blind reconstruction: 14.1% band-max. Under the truth-referenced two-parameter
(SPAM + decay) calibration: **0.19% band-max against a 0.12% exact-moment truncation floor**, gain-pole
weight 0.97, and the passive fit beat the kernel sum on identical moments in every arm. The device's SPAM
prefactor (5.7% / 3.3% per port) and per-gate retention (0.9945) are now separately measured constants;
the wall estimate updates to ≈870 routed gates once SPAM is separated from decay.

## 0. Control panel — everything you set is here

One switch and one cell. `RUN_ON_HARDWARE = False` runs the identical experiment on a local noisy
simulator (validate the flow this way first, it costs nothing). Flip it to `True` to submit one batched
job to an IBM QPU. First hardware run only: paste your 44-character API key
([quantum.cloud.ibm.com](https://quantum.cloud.ibm.com) → Create API key) and, optionally, your Open-Plan
instance CRN (dashboard → **Instances** tab — the same CRN you used in the QSP notebook); the cell saves
them to `~/.qiskit/qiskit-ibm.json`, after which you should **blank the token here**. The cell also
installs all dependencies (three-rung pip ladder) and, in hardware mode, lists your reachable backends as
a sanity check before anything is built.

**New controls:** `INSTANCE` — `"lite"` (default; shallow, survives today's noise) or `"deep"` (the
4-state wall probe). `REANALYZE_JSON` — point it at any previous `qrl_hw_results*.json` to re-run the
gated analysis on that run's stored moments without submitting anything (set `INSTANCE` to match the
file).

In [ ]:
# ============================ CONTROL PANEL ============================
RUN_ON_HARDWARE = False   # True -> submit ONE batched job to an IBM QPU (~1-2 min of the
                          #         10 free Open-Plan minutes); False -> local noisy simulator
IBM_API_TOKEN   = ""      # 44-char key from https://quantum.cloud.ibm.com (needed once;
                          #         saved to ~/.qiskit/qiskit-ibm.json, then blank it here)
IBM_INSTANCE    = ""      # optional: your Open-Plan CRN "crn:v1:..." (same as your QSP notebook)
IBM_BACKEND     = ""      # optional: pin a device, e.g. "ibm_fez"; blank = least busy
INSTANCE        = "lite"  # "lite": 2-state chain, ~10-25 routed 2q/step -> several moments survive
                          # "deep": 4-state bottlenecked chain, ~120-155 2q/step -> wall probe
REANALYZE_JSON  = ""      # path to a previous qrl_hw_results*.json: skip all circuits and re-run
                          #         the gated analysis on that run's stored moments (INSTANCE must match)
SHOTS           = 8192    # per circuit
K_MAX           = 10      # highest Chebyshev moment (deepest circuit = K_MAX walk steps)
FORCE_REINSTALL = False   # True (then RESTART the kernel) to force a matched-latest qiskit stack

# dependencies: three-rung ladder (plain -> --break-system-packages -> --ignore-installed)
import importlib.util, subprocess, sys
PKGS = ["numpy", "scipy", "matplotlib", "qiskit", "qiskit-aer", "qiskit-ibm-runtime"]
def _pip(a):
    return subprocess.run([sys.executable, "-m", "pip", "install", "-q", *a]).returncode
_need = PKGS if FORCE_REINSTALL else [p for p in PKGS
                                      if importlib.util.find_spec(p.replace("-", "_")) is None]
if _need:
    _args = (["-U"] if FORCE_REINSTALL else []) + _need
    _rc = _pip(_args)
    if _rc:
        _rc = _pip(["--break-system-packages", *_args])
    if _rc:
        _rc = _pip(["--break-system-packages", "--ignore-installed", *_args])
    print("install exit:", _rc, "| installed:", _need)
    if FORCE_REINSTALL:
        print(">>> RESTART the kernel now, then re-run from the top <<<")
else:
    print("dependencies present")

# IBM account: saved once when a token is pasted; sanity check only in hardware mode
if IBM_API_TOKEN:
    from qiskit_ibm_runtime import QiskitRuntimeService
    _kw = dict(token=IBM_API_TOKEN, set_as_default=True, overwrite=True)
    if IBM_INSTANCE:
        _kw["instance"] = IBM_INSTANCE
    QiskitRuntimeService.save_account(**_kw)
    print("credentials saved — you can blank IBM_API_TOKEN now")
if RUN_ON_HARDWARE:
    from qiskit_ibm_runtime import QiskitRuntimeService
    _s = QiskitRuntimeService(**({"instance": IBM_INSTANCE} if IBM_INSTANCE else {}))
    print("backends:", [b.name for b in _s.backends(operational=True, simulator=False)])
print("mode:", "IBM HARDWARE" if RUN_ON_HARDWARE else "local noisy simulator")

**Troubleshooting** (each hit in live runs of the sibling notebooks):

- `error 1352: not authorized to run a session ... open plan` — cannot occur here: this notebook submits
  **one job** (`SamplerV2(mode=backend)`), never a Session.
- `TranspilerError: Invalid plugin name ibm_dynamic_circuits` — qiskit / qiskit-ibm-runtime version skew.
  Set `FORCE_REINSTALL = True` in the control panel, **restart the kernel**, re-run; the run cell also
  self-heals through fallback pass managers.
- `ModuleNotFoundError` on a fresh runtime — the control panel installs everything; it escalates through
  `--break-system-packages` and `--ignore-installed` for distro-owned conflicts (e.g. PyJWT).
- `AccountNotFoundError` — only possible with `RUN_ON_HARDWARE = True` and no saved credentials: paste the
  token once in the control panel.
- **Wall mode** (`NOISE-CHARACTERIZATION`) — not an error: the run measured the decoherence wall (fewer
  than 3 moment orders above $5\sigma$). The curve table is suppressed by design; switch to
  `INSTANCE = "lite"` for spectroscopy.

## 1. Protocol constants and imports

Fixed protocol — nothing here needs editing.

In [ ]:
import numpy as np, json, time
from numpy.polynomial import chebyshev as C
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit.library import PauliGate, StatePreparation, DiagonalGate
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
from scipy.optimize import nnls

RUNGS  = [0.4, 0.125]     # 1 - gamma at the two rungs (H = 2.5, 8)
BAND_H = (1.5, 8.0)       # evaluation band in horizon units
N_BAND = 21
SEED   = 7

## 2. Testbed — two instance tiers

Both are the paper's two-port construction. **lite**: a 2-state lazy chain, spec$(S)=\{0.5, 1\}$,
one system qubit and one LCU ancilla — about 6 two-qubit gates per walk step, so ten Chebyshev orders fit
under today's decoherence wall. **deep**: the 4-state bottlenecked chain of the original design — a wall
probe on current devices (run 1 on `ibm_fez`: only $k=1$ survived).

In [ ]:
def build_instance(name):
    if name == "deep":
        w = np.array([1.0, 0.02, 1.0])                  # bottleneck mid
        A = np.zeros((4, 4))
        for i, wi in enumerate(w):
            A[i, i+1] = A[i+1, i] = wi
        beta = 0.5
        d0 = np.array([1.0, 0, 0, 0]); r = np.array([1.3, 0.7, 1.1, 2.0])
    elif name == "lite":
        A = np.array([[0.0, 1.0], [1.0, 0.0]])
        beta = 0.25                                     # eigs {1, 1-2*beta} = {1, 0.5}
        d0 = np.array([1.0, 0.0]); r = np.array([0.6, 1.7])
    else:
        raise ValueError(name)
    deg = A.sum(1)
    S_ = (1-beta)*np.eye(len(A)) + beta*(A/np.sqrt(np.outer(deg, deg)))
    return S_, d0/np.sqrt(deg), np.sqrt(deg)*r

S, u, v = build_instance(INSTANCE)
N = len(S)
NSYS = int(np.log2(N))
sx = np.linalg.norm(u)*np.linalg.norm(v)
uh, vh = u/np.linalg.norm(u), v/np.linalg.norm(v)
rho = float(uh @ vh)
Sb = {+1: 2+2*rho, -1: 2-2*rho}
bport = {sg: (uh + sg*vh)/np.sqrt(Sb[sg]) for sg in (+1, -1)}
print(f"instance '{INSTANCE}': N={N}, spec(S)={np.round(np.linalg.eigvalsh(S), 4)}")

## 3. Block encoding and the qubitized walk

$S=\sum_j c_j P_j$ (all-positive here), PREPARE/SELECT with $2$ ancillas, subnormalization
$\lambda_1=\sum_j|c_j|$; the walk $W=R_{|0\rangle}\,(\mathrm{PREP}^{\dagger}\,\mathrm{SELECT}\,\mathrm{PREP})$
satisfies $\mathrm{Re}\,\langle 0,b|W^k|0,b\rangle=\hat b^{\top}T_k(S/\lambda_1)\,\hat b$ **exactly** — the
Chebyshev property of qubitization — measured by a Hadamard test on one control qubit
(total: $1+2+2=5$ qubits).

In [ ]:
op = SparsePauliOp.from_operator(S).simplify()
labels = [str(p) for p in op.paulis]; coefs = np.real(op.coeffs)
keep = np.abs(coefs) > 1e-12
labels = [l for l, k in zip(labels, keep) if k]; coefs = coefs[keep]
lam1 = float(np.abs(coefs).sum())                   # subnormalization
m_anc = int(np.ceil(np.log2(len(labels)))) if len(labels) > 1 else 1
NT = 2**m_anc
amps = np.zeros(NT); amps[:len(coefs)] = np.sqrt(np.abs(coefs)/lam1)
amps /= np.linalg.norm(amps)
signs = np.ones(NT); signs[:len(coefs)] = np.sign(coefs)
D_eff = S/lam1                                      # what the walk's moments see
x_atom = 1.0/lam1                                   # exact stationary atom of D_eff
print(f"LCU: {len(labels)} Pauli terms {labels}, lam1={lam1:.6f}, ancillas={m_anc}")

def make_prep():
    qc = QuantumCircuit(m_anc); qc.append(StatePreparation(amps), range(m_anc)); return qc
PREP = make_prep()

def append_select_sd(qc, anc, sys, extra_ctrl=None):
    """SELECT then sign-diagonal, optionally with one extra control qubit.
    The all-identity term contributes nothing; a trivial sign diagonal is skipped."""
    for j, lab in enumerate(labels):
        if set(lab) == {"I"}:
            continue                       # controlled identity: no gate needed
        g = PauliGate(lab)
        nctl = m_anc + (1 if extra_ctrl is not None else 0)
        cs = j if extra_ctrl is None else (j | (1 << m_anc))
        qargs = (list(anc) + ([extra_ctrl] if extra_ctrl is not None else []) + list(sys))
        qc.append(g.control(nctl, ctrl_state=cs), qargs)
    if np.any(signs < 0):
        diag = signs.astype(complex)
        if extra_ctrl is None:
            qc.append(DiagonalGate(diag), anc)
        else:
            qc.append(DiagonalGate(np.concatenate([np.ones(NT), diag])), list(anc)+[extra_ctrl])

def append_refl(qc, anc, extra_ctrl=None):
    """R = 2|0..0><0..0| - I on the ancillas (global-phase-exact); for m_anc=2,
    R = Z_a0 Z_a1 CZ_a0a1, and the controlled version replaces each factor by
    its controlled counterpart."""
    if m_anc == 2:
        a0, a1 = anc
        if extra_ctrl is None:
            qc.z(a0); qc.z(a1); qc.cz(a0, a1)
        else:
            qc.cz(extra_ctrl, a0); qc.cz(extra_ctrl, a1); qc.ccz(extra_ctrl, a0, a1)
    else:
        dref = -np.ones(NT, complex); dref[0] = 1.0
        if extra_ctrl is None:
            qc.append(DiagonalGate(dref), anc)
        else:
            qc.append(DiagonalGate(np.concatenate([np.ones(NT), dref])), list(anc)+[extra_ctrl])

def walk_once(qc, anc, sys, ctrl=None):
    # W = R_{|0>} . (PREP^dag SELECT.SD PREP): first-appended acts first
    qc.append(PREP, anc)
    append_select_sd(qc, anc, sys, extra_ctrl=ctrl)
    qc.append(PREP.inverse(), anc)
    append_refl(qc, anc, extra_ctrl=ctrl)

def hadamard_test(k, port, measured=True):
    """ctrl + m_anc ancillas + 2 system qubits; returns Re<0,b|W^k|0,b> via <Z>_ctrl."""
    nq = 1 + m_anc + NSYS
    qc = QuantumCircuit(nq, 1 if measured else 0)
    ctrl, anc, sys = 0, list(range(1, 1+m_anc)), list(range(1+m_anc, nq))
    qc.append(StatePreparation(bport[port]), sys)
    qc.h(ctrl)
    for _ in range(k):
        walk_once(qc, anc, sys, ctrl=ctrl)
    qc.h(ctrl)
    if measured:
        qc.measure(ctrl, 0)
    return qc

### 3a. Correctness gate

Before anything is run on hardware, the construction must reproduce the Chebyshev identity to numerical
precision on a statevector. If this cell raises, do not proceed.

In [ ]:
def _Tk(M, k):
    if k == 0: return np.eye(len(M))
    if k == 1: return M
    Tm2, Tm1 = np.eye(len(M)), M
    for _ in range(2, k+1):
        Tm2, Tm1 = Tm1, 2*M@Tm1 - Tm2
    return Tm1
ok = True
for port in (+1, -1):
    for k in range(0, 5):
        qc = hadamard_test(k, port, measured=False)
        sv = Statevector.from_instruction(qc)
        pz = 0.0
        for idx, a in enumerate(sv.data):
            pz += (1 if (idx & 1) == 0 else -1) * abs(a)**2
        mu_ex = float(bport[port] @ _Tk(D_eff, k) @ bport[port])
        if abs(pz - mu_ex) > 1e-9: ok = False; print("MISMATCH", port, k, pz, mu_ex)
assert ok, "walk construction failed Chebyshev identity"
print("validation: Re<Z>_ctrl == b^T T_k(S/lam1) b  for k=0..4, both ports  [OK]")

## 4. Run

All $2\times K$ circuits are submitted as **one batched job** (dynamical decoupling and gate twirling
enabled on hardware — twirling also makes the depolarizing calibration model below well-founded).

With `REANALYZE_JSON` set, this cell skips all circuits and loads the stored moments instead — the gated
analysis below then runs unchanged on the old data.

In [ ]:
ks = list(range(1, K_MAX+1))
if REANALYZE_JSON:
    prev = json.load(open(REANALYZE_JSON))
    K_MAX = prev["K_max"]; ks = list(range(1, K_MAX+1))
    mu_hw = {sg: np.array(prev["mu_hw"][str(sg)]) for sg in (+1, -1)}
    mu_sd = {sg: np.array(prev["mu_sd"][str(sg)]) for sg in (+1, -1)}
    twoq = {k: prev["twoq"][str(k)] for k in ks}
    backend_name = prev["backend"] + " (reanalysis)"
    hardware_mode = bool(prev.get("hardware", True))
    _ex_file = {sg: np.array(prev["mu_exact"][str(sg)]) for sg in (+1, -1)}
    print(f"REANALYSIS of {REANALYZE_JSON}: {backend_name}, K_max={K_MAX}")
elif not RUN_ON_HARDWARE:
    pubs = [(port, k, hadamard_test(k, port)) for port in (+1, -1) for k in ks]
    from qiskit_aer import AerSimulator
    from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
    nm = NoiseModel()
    nm.add_all_qubit_quantum_error(depolarizing_error(3e-4, 1), ["u", "rz", "sx", "x", "h"])
    nm.add_all_qubit_quantum_error(depolarizing_error(3e-3, 2), ["cx", "cz", "ecr"])
    nm.add_all_qubit_readout_error(ReadoutError([[0.99, 0.01], [0.01, 0.99]]))
    backend = AerSimulator(noise_model=nm, seed_simulator=SEED)
    pm = generate_preset_pass_manager(backend=backend, optimization_level=3, seed_transpiler=SEED)
    isa = [pm.run(qc) for _, _, qc in pubs]
    twoq = {}
    for (port, k, _), c in zip(pubs, isa):
        n2 = sum(cnt for g, cnt in c.count_ops().items() if g in ("cx", "cz", "ecr"))
        twoq.setdefault(k, n2)
    job = backend.run(isa, shots=SHOTS)
    counts = job.result().get_counts()
    if isinstance(counts, dict): counts = [counts]
    res = {}
    for (port, k, _), ct in zip(pubs, counts):
        p0 = ct.get("0", 0)/SHOTS; z = 2*p0 - 1
        res[(port, k)] = (float(z), float(np.sqrt(max(1-z*z, 1e-12)/SHOTS)))
    backend_name = "aer(depol 2q=0.3%,1q=0.03%,ro=1%)"
else:  # RUN_ON_HARDWARE
    pubs = [(port, k, hadamard_test(k, port)) for port in (+1, -1) for k in ks]
    import warnings; warnings.filterwarnings("ignore")
    from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
    service = QiskitRuntimeService(**({"instance": IBM_INSTANCE} if IBM_INSTANCE else {}))
    used = 0.0
    for j in service.jobs(limit=20):
        try:
            used += j.metrics()["usage"].get("quantum_seconds", 0.0)
        except Exception:
            pass
    print(f"QPU used recently: ~{used:.0f} s of the ~600 s Open-plan window")
    if used > 480:
        print("WARNING: window nearly spent — this 1-2 min run may exceed it; "
              "consider waiting for the 28-day reset.")
    need = 1 + m_anc + NSYS
    if IBM_BACKEND:
        backend = service.backend(IBM_BACKEND)
    else:
        backend = service.least_busy(operational=True, simulator=False, min_num_qubits=need)
    print("backend:", backend.name)
    import qiskit as _qk, qiskit_ibm_runtime as _qir
    print("qiskit", _qk.__version__, "| qiskit-ibm-runtime", _qir.__version__,
          "(on 'Invalid plugin name': FORCE_REINSTALL=True in the install cell, restart kernel)")
    def make_pm(be):
        try:
            return generate_preset_pass_manager(backend=be, optimization_level=3,
                                                seed_transpiler=SEED)
        except Exception as e1:
            print("default pass manager failed:", e1)
        try:
            return generate_preset_pass_manager(target=be.target, optimization_level=3,
                                                translation_method="translator",
                                                seed_transpiler=SEED)
        except Exception as e2:
            print("target pass manager failed:", e2)
        cfg = be.configuration()
        return generate_preset_pass_manager(optimization_level=3, seed_transpiler=SEED,
                                            basis_gates=cfg.basis_gates,
                                            coupling_map=be.coupling_map)
    pm = make_pm(backend)
    isa = [pm.run(qc) for _, _, qc in pubs]
    twoq = {}
    for (port, k, _), c in zip(pubs, isa):
        n2 = sum(cnt for g, cnt in c.count_ops().items() if g in ("cx", "cz", "ecr"))
        twoq.setdefault(k, n2)
    sampler = SamplerV2(mode=backend)
    sampler.options.default_shots = SHOTS
    try:
        sampler.options.dynamical_decoupling.enable = True
        sampler.options.twirling.enable_gates = True
    except Exception as e:
        print("note: mitigation options unavailable on this runtime version:", e)
    job = sampler.run(isa)
    print("job id:", job.job_id())
    out = job.result()
    res = {}
    for (port, k, _), pr in zip(pubs, out):
        ct = pr.data.c.get_counts()
        tot = sum(ct.values()); p0 = ct.get("0", 0)/tot; z = 2*p0 - 1
        res[(port, k)] = (float(z), float(np.sqrt(max(1-z*z, 1e-12)/tot)))
    backend_name = backend.name

if not REANALYZE_JSON:
    mu_hw = {port: np.array([1.0] + [res[(port, k)][0] for k in ks]) for port in (+1, -1)}
    mu_sd = {port: np.array([0.0] + [res[(port, k)][1] for k in ks]) for port in (+1, -1)}
    hardware_mode = bool(RUN_ON_HARDWARE)
mu_ex_arr = {port: np.array([float(bport[port] @ _Tk(D_eff, k) @ bport[port]) for k in range(K_MAX+1)])
             for port in (+1, -1)}
if REANALYZE_JSON:
    for sg in (+1, -1):
        assert np.allclose(mu_ex_arr[sg], _ex_file[sg], atol=1e-8), \
            "REANALYZE_JSON was produced by a different INSTANCE — set INSTANCE to match the file"

## 5. SNR gate, Q1, and calibration

Every downstream claim is gated: a moment order is **informative** only if it clears
$5\sigma$ of shot noise *with the right sign* (a decohered value pushed over the line by a dead-tail
offset does not count). Fewer than 3 informative orders on either port ⇒ **noise-characterization mode**:
the run is reported as a decoherence-wall measurement, the retention fit uses informative orders only, and
the calibration abstains — below the floor there is no signal to rescale, and amplifying dead moments
manufactures curves (run 1's calibrated arm reached $-218$ on a moment whose true value was $0.11$).

The calibration is **two-parameter and fitted per port**:
$\ln(\mu_{\rm meas}/\mu_{\rm exact}) = a + b\,n_{2q}$ — a depth-independent SPAM prefactor $e^{a}$
(state-preparation-and-measurement loss; run 2 measured $5.7\%$ on port $+$ and $3.3\%$ on port $-$,
which the earlier one-parameter model silently folded into the slope) plus a per-two-qubit-gate decay
$e^{b}$. It is applied only to informative orders; below-floor orders are shrunk to zero, never amplified.
**Label it honestly wherever used:** the parameters are fitted against the known exact reference, so
calibrated numbers are *validation-only* — they answer "does a two-parameter noise model explain the
data," not "what would a blind run achieve." The raw numbers are the blind result.

In [ ]:
INFORMATIVE_SIGMA = 5.0
def _informative(sg):
    out = []
    for k in ks:
        sd_k = max(mu_sd[sg][k], 1e-12)
        strong = abs(mu_hw[sg][k]) > INFORMATIVE_SIGMA*sd_k
        right_sign = (mu_hw[sg][k]*mu_ex_arr[sg][k] > 0) or \
                     (abs(mu_ex_arr[sg][k]) < INFORMATIVE_SIGMA*sd_k)
        if strong and right_sign:
            out.append(k)
    return out
info = {sg: _informative(sg) for sg in (+1, -1)}
n_info = min(len(info[+1]), len(info[-1]))
SPECTROSCOPY = n_info >= 3
dead = {sg: [k for k in ks if k not in info[sg]] for sg in (+1, -1)}
offset = {sg: (float(np.mean([mu_hw[sg][k] for k in dead[sg]])) if dead[sg] else 0.0)
          for sg in (+1, -1)}
print(f"\nSNR gate ({INFORMATIVE_SIGMA:.0f}-sigma, sign-consistent): informative orders "
      f"port+ {info[+1]}  port- {info[-1]}  ->  mode: "
      + ("SPECTROSCOPY" if SPECTROSCOPY else "NOISE CHARACTERIZATION"))

In [ ]:
print(f"\nQ1 — per-moment hardware error vs circuit volume (port +):")
print("  k   2q-gates   mu_exact    mu_meas     bias        shot-sd")
for k in ks:
    b = mu_hw[+1][k]-mu_ex_arr[+1][k]
    print(f"{k:4d} {str(twoq[k]):>9} {mu_ex_arr[+1][k]:+.5f} {mu_hw[+1][k]:+.5f} "
          f"{b:+.5f}  {mu_sd[+1][k]:.5f}")
fitp = {}
for sg in (+1, -1):
    f_ks = [k for k in info[sg] if abs(mu_ex_arr[sg][k]) > 0.05]
    if len(f_ks) >= 2:
        dpt = np.array([twoq[k] for k in f_ks], float)
        rat = np.array([mu_hw[sg][k]/mu_ex_arr[sg][k] for k in f_ks])
        b_, a_ = np.polyfit(dpt, np.log(rat), 1)     # ln(ratio) = a + b*n2q
        fitp[sg] = (float(a_), float(b_))
    elif len(f_ks) == 1:
        k0 = f_ks[0]
        fitp[sg] = (0.0, float(np.log(mu_hw[sg][k0]/mu_ex_arr[sg][k0])/twoq[k0]))
        print(f"  port {sg:+d}: single informative order (k={k0}) — SPAM prefactor not "
              f"separable from decay (one-point estimate)")
    else:
        fitp[sg] = None
slope_cal = fitp[+1][1] if fitp[+1] is not None else None
for sg in (+1, -1):
    if fitp[sg] is not None and len([k for k in info[sg] if abs(mu_ex_arr[sg][k]) > 0.05]) >= 2:
        a_, b_ = fitp[sg]
        print(f"  port {sg:+d}: retention {np.exp(b_):.5f}/2q-gate; SPAM prefactor "
              f"e^a = {np.exp(a_):.4f} (depth-independent loss {100*(1-np.exp(a_)):.1f}%)")
wall_gates = None
if slope_cal is not None and slope_cal < 0:
    sd_typ = float(np.median([mu_sd[+1][k] for k in ks]))
    mu_typ = float(np.median([abs(mu_ex_arr[+1][k]) for k in ks]))
    a0 = fitp[+1][0]
    wall_gates = float(np.log(mu_typ*np.exp(a0)/max(sd_typ, 1e-12))/(-slope_cal))
    print(f"  decoherence wall (typical signal {mu_typ:.2f} x e^a -> shot noise "
          f"{sd_typ:.3f}): ~{wall_gates:.0f} routed 2q gates")
for sg in (+1, -1):
    if dead[sg]:
        zs = np.array([mu_hw[sg][k]/max(mu_sd[sg][k], 1e-12) for k in dead[sg]])
        print(f"  dead-tail additive offset port {sg:+d}: mean {offset[sg]:+.4f} "
              f"({np.mean(zs):+.2f} sigma/pt over {len(dead[sg])} orders; "
              f"aggregate {np.mean(zs)*np.sqrt(len(dead[sg])):+.1f} sigma)")

In [ ]:
mu_cal = None
if SPECTROSCOPY and all(fitp[sg] is not None and fitp[sg][1] < 0 for sg in (+1, -1)):
    mu_cal = {}
    for sg in (+1, -1):
        a_, b_ = fitp[sg]
        arr = np.zeros(K_MAX+1); arr[0] = 1.0
        for k in ks:
            if k in info[sg]:
                arr[k] = mu_hw[sg][k]*np.exp(-(a_ + b_*twoq[k]))
            # below-floor orders shrunk to 0: the decay model has no signal there to rescale
        mu_cal[sg] = arr
    n_shrunk = sum(len(dead[sg]) for sg in (+1, -1))
    print(f"\ncalibration: two-parameter (SPAM prefactor + per-2q-gate decay), fitted per "
          f"port, applied to informative orders; {n_shrunk} below-floor orders shrunk to 0 "
          f"(never amplified)")
else:
    print("\ncalibration: ABSTAINS — "
          + ("no usable retention fit" if slope_cal is None else
             "noise-characterization mode (rescaling dead moments manufactures signal)"))

## 6. Q2 — value-curve reconstruction (gated)

The exact-moment control arm (the truncation floor) always prints. Hardware curve tables print **only in
spectroscopy mode**; in wall mode they are suppressed with an explicit banner, and the saved JSON records
`"mode": "noise_characterization"` with the wall summary instead of accuracy numbers.

In [ ]:
Hs = np.geomspace(*BAND_H, N_BAND); gams = 1 - 1/Hs
def kernel_coeffs(gam, K, M=4096):
    th = (np.arange(M)+0.5)*np.pi/M; x = np.cos(th)
    f = 1.0/(1.0 - gam*lam1*x)
    a = np.array([2/M*np.sum(f*np.cos(k*th)) for k in range(K+1)]); a[0] /= 2
    return a
def J_exact(gam):
    Rm = np.linalg.inv(np.eye(N) - gam*S)
    return float(u @ Rm @ v)
def J_from_mu(gam, mu):
    a = kernel_coeffs(gam, K_MAX)
    Jp = {sg: float(a @ mu[sg]) for sg in (+1, -1)}
    return sx/4*(Sb[+1]*Jp[+1] - Sb[-1]*Jp[-1])
deltas = np.geomspace(3e-2, 1.6, 48)   # keep grid poles separated from the exact atom
xs = np.concatenate([[x_atom], x_atom*(1-deltas)])
Adict = np.array([C.chebval(xs, np.eye(K_MAX+1)[k]) for k in range(K_MAX+1)])
Wrow = np.ones(K_MAX+1); Wrow[0] = 100.0
def passive_fit(mu_vec):
    wgt, _ = nnls(Adict*Wrow[:, None], mu_vec*Wrow)
    return wgt
def J_from_fit(gam, wf):
    return {sg: float(np.sum(wf[sg]/(1 - gam*lam1*xs))) for sg in (+1, -1)}
def J_model(gam, wf):
    Jp = J_from_fit(gam, wf)
    return sx/4*(Sb[+1]*Jp[+1] - Sb[-1]*Jp[-1])

# exact-moment control arm: same estimator layer fed noiseless moments
wfit_ex = {sg: passive_fit(mu_ex_arr[sg]) for sg in (+1, -1)}
Jex = np.array([J_exact(g) for g in gams])
Jks_ex = np.array([J_from_mu(g, mu_ex_arr) for g in gams])
Jmd_ex = np.array([J_model(g, wfit_ex) for g in gams])
err_ks_ex = np.abs(Jks_ex/Jex - 1); err_md_ex = np.abs(Jmd_ex/Jex - 1)
print(f"\nQ2 — curve reconstruction from {backend_name} moments over "
      f"H in [{BAND_H[0]}, {BAND_H[1]}]:")
print(f"  exact-moment control:    kernel sum band-max {err_ks_ex.max()*100:.3f}%  |  "
      f"pole model band-max {err_md_ex.max()*100:.4f}%")

err_ks = err_md = err_ksc = err_mdc = None
if SPECTROSCOPY:
    wfit_hw = {sg: passive_fit(mu_hw[sg]) for sg in (+1, -1)}
    Jks = np.array([J_from_mu(g, mu_hw) for g in gams])
    Jmd = np.array([J_model(g, wfit_hw) for g in gams])
    err_ks = np.abs(Jks/Jex - 1); err_md = np.abs(Jmd/Jex - 1)
    print(f"  raw        kernel sum:   band-max {err_ks.max()*100:.3f}%   "
          f"median {np.median(err_ks)*100:.3f}%")
    print(f"  raw        pole model:   band-max {err_md.max()*100:.3f}%   "
          f"median {np.median(err_md)*100:.3f}%")
    if mu_cal is not None:
        wfit_cal = {sg: passive_fit(mu_cal[sg]) for sg in (+1, -1)}
        Jksc = np.array([J_from_mu(g, mu_cal) for g in gams])
        Jmdc = np.array([J_model(g, wfit_cal) for g in gams])
        err_ksc = np.abs(Jksc/Jex - 1); err_mdc = np.abs(Jmdc/Jex - 1)
        print(f"  calibrated kernel sum:   band-max {err_ksc.max()*100:.3f}%   "
              f"median {np.median(err_ksc)*100:.3f}%")
        print(f"  calibrated pole model:   band-max {err_mdc.max()*100:.3f}%   "
              f"median {np.median(err_mdc)*100:.3f}%")
        npoles = {sg: int(np.sum(wfit_cal[sg] > 1e-9)) for sg in (+1, -1)}
        print(f"  active poles: {npoles}, gain-pole weights: "
              f"{wfit_cal[+1][0]:.4f} / {wfit_cal[-1][0]:.4f}")
        verdict = ("passive fit <= kernel sum (distillation survives this noise)"
                   if err_mdc.max() <= err_ksc.max()
                   else "kernel sum < passive fit at this depth/noise (report as measured)")
        print("  verdict (calibrated arm):", verdict)
else:
    print("  " + "="*68)
    print(f"  NOISE-CHARACTERIZATION MODE (n_informative = {len(info[+1])}/{len(info[-1])} < 3).")
    print("  This run measures the decoherence wall, not spectroscopy. Hardware")
    print("  curve reconstruction is SUPPRESSED and must not be quoted as accuracy.")
    if wall_gates is not None:
        print(f"  Wall summary: retention {np.exp(slope_cal):.5f}/2q-gate; "
              f"signal reaches shot noise at ~{wall_gates:.0f} routed 2q gates.")
    print("  Variance validation stands: shot-sd matches sqrt((1-mu^2)/shots).")
    print("  To do spectroscopy on this device, use INSTANCE='lite' (shallow walk).")
    print("  " + "="*68)

json.dump({"backend": backend_name, "hardware": bool(hardware_mode),
           "instance": INSTANCE, "mode": ("spectroscopy" if SPECTROSCOPY
                                          else "noise_characterization"),
           "K_max": K_MAX, "shots": SHOTS, "lam1": lam1, "labels": labels,
           "informative": {str(sg): info[sg] for sg in info},
           "n_informative": n_info,
           "retention_per_2q": (float(np.exp(slope_cal)) if slope_cal is not None else None),
           "wall_gates": wall_gates,
           "dead_tail_offset": {str(sg): offset[sg] for sg in offset},
           "twoq": {str(k): twoq[k] for k in ks},
           "mu_hw": {str(sg): mu_hw[sg].tolist() for sg in mu_hw},
           "mu_sd": {str(sg): mu_sd[sg].tolist() for sg in mu_sd},
           "mu_exact": {str(sg): mu_ex_arr[sg].tolist() for sg in mu_ex_arr},
           "err_kernel_sum": (err_ks.tolist() if err_ks is not None else None),
           "err_pole_model": (err_md.tolist() if err_md is not None else None),
           "err_kernel_sum_cal": (err_ksc.tolist() if err_ksc is not None else None),
           "err_pole_model_cal": (err_mdc.tolist() if err_mdc is not None else None),
           "err_kernel_sum_exact": err_ks_ex.tolist(),
           "err_pole_model_exact": err_md_ex.tolist(),
           "H_band": Hs.tolist(), "timestamp": time.strftime("%Y-%m-%d %H:%M:%S")},
          open("qrl_hw_results.json", "w"), indent=1)
print("\nsaved qrl_hw_results.json")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(9.6, 3.4))
d = np.array([twoq[k] for k in ks], float)
ax[0].plot(d, [mu_ex_arr[+1][k] for k in ks], "k.-", label="exact")
ax[0].errorbar(d, [mu_hw[+1][k] for k in ks],
               yerr=[mu_sd[+1][k] for k in ks], fmt="o", ms=4, label="measured")
band = INFORMATIVE_SIGMA*np.array([mu_sd[+1][k] for k in ks])
ax[0].fill_between(d, -band, band, color="0.85", label=f"±{INFORMATIVE_SIGMA:.0f}σ shot band")
if mu_cal is not None:
    ax[0].plot(d, [mu_cal[+1][k] for k in ks], "s--", ms=4, label="calibrated (informative)")
ax[0].set_xlabel("transpiled two-qubit gates"); ax[0].set_ylabel(r"$\mu_k$ (port $+$)")
ax[0].legend(fontsize=7.5); ax[0].set_title("Q1: moment signal vs circuit volume", fontsize=9)
ax[1].semilogy(Hs, np.maximum(err_md_ex, 1e-8)*100, "k--", lw=1,
               label="exact-moment pole model (floor)")
if SPECTROSCOPY:
    if err_ksc is not None:
        ax[1].semilogy(Hs, np.maximum(err_ksc, 1e-6)*100, "o-", ms=4,
                       label="calibrated kernel sum")
        ax[1].semilogy(Hs, np.maximum(err_mdc, 1e-6)*100, "s-", ms=4,
                       label="calibrated pole model")
    else:
        ax[1].semilogy(Hs, np.maximum(err_ks, 1e-6)*100, "o-", ms=4, label="raw kernel sum")
        ax[1].semilogy(Hs, np.maximum(err_md, 1e-6)*100, "s-", ms=4, label="raw pole model")
else:
    ax[1].text(0.5, 0.6, "noise-characterization mode\n"
               f"(n_informative = {len(info[+1])}/{len(info[-1])})\n"
               "curve reconstruction suppressed",
               transform=ax[1].transAxes, ha="center", va="center", fontsize=9,
               bbox=dict(boxstyle="round", fc="mistyrose", ec="firebrick"))
ax[1].set_xlabel("horizon H"); ax[1].set_ylabel("|rel. error| (%)")
ax[1].legend(fontsize=7.5); ax[1].set_title("Q2: value-curve reconstruction", fontsize=9)
fig.tight_layout(); fig.savefig("qrl_hw_diagnostics.png", dpi=160); plt.show()
print("saved qrl_hw_diagnostics.png")
try:
    from google.colab import files
    files.download("qrl_hw_results.json"); files.download("qrl_hw_diagnostics.png")
except Exception:
    pass

## 7. How this enters the paper

The two-run record on `ibm_fez` is the hardware story:

> *Run 1 (deep instance, 123–1547 routed two-qubit gates) measured the device, not the chain: only $k=1$
> cleared the $5\sigma$ gate (retention $0.9917$ per routed gate, port-consistent to $3\times10^{-5}$),
> locating the decoherence wall near $500$ routed gates, with per-moment shot variance matching
> $\sqrt{(1-\mu^2)/\mathrm{shots}}$ throughout — validating the variance half of the paper's noise proxy
> and replacing its bias half with measured multiplicative decay. Run 2 (shallow two-state instance,
> 7–97 routed gates) performed genuine spectroscopy: all ten Chebyshev orders on both balanced ports
> cleared the sign-consistent $5\sigma$ gate, sign-perfect on the oscillating port, and the paper's
> estimator layer reconstructed the value–discount curve to $14.1\%$ band-max blind and $0.19\%$ under a
> truth-referenced two-parameter (SPAM $+$ decay) calibration — against a $0.12\%$ exact-moment
> truncation floor — with the stationary gain-pole weight identifiable at $0.97$ and the passive fit
> beating the kernel sum on identical moments in every arm. The calibration is validation-only; no
> advantage is claimed at this scale, and the deep-horizon conditioning regime remains simulation-only.*

**Bars (both runs):** wall-mode curve numbers are never quoted as accuracy (run 1's "11.5%" is the
cautionary example, and it is barred); calibrated numbers always carry the truth-referenced /
validation-only label next to the blind raw numbers.